# FacePred Colab Quickstart

This notebook prepares a cheap metadata-derived cache, trains `FacePredWorldModel`, checkpoints to Google Drive, and evaluates the best checkpoint.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# Update these for your repo and Drive layout.
REPO_URL = 'https://github.com/YOUR_USER/YOUR_REPO.git'
DRIVE_ROOT = '/content/drive/MyDrive/facepred'
PROJECT_DIR = '/content/facepred'
CACHE_NAME = 'meld_cheap_v0'
RUN_NAME = 'world_xs_cheap_v0'

In [ ]:
!cd /content && (git clone $REPO_URL facepred || true)
%cd /content/facepred
!git pull

In [ ]:
# Minimal deps for cheap-cache training. Do not overwrite Colab's CUDA torch.
!pip install -e .
!pip install numpy==1.26.4 pandas scipy pyyaml hydra-core omegaconf rich pytest ruff

In [ ]:
!python -m ruff check .
!python -m pytest -q

## Prepare Cache

Use the real MELD command if you have raw MELD under Drive. Use the synthetic command for a pipeline smoke test.

In [ ]:
# Real MELD metadata cache.
!python scripts/prepare_meld_cache.py \
  --config configs/config.yaml \
  --data-root "$DRIVE_ROOT/data/MELD.Raw" \
  --output-dir "$DRIVE_ROOT/cache/$CACHE_NAME" \
  --splits train dev test

In [ ]:
# If you do not have MELD ready yet, use this synthetic smoke cache instead.
# CACHE_NAME = 'synthetic_cheap_v0'
# !python scripts/prepare_meld_cache.py \
#   --config configs/config.yaml \
#   --output-dir "$DRIVE_ROOT/cache/$CACHE_NAME" \
#   --synthetic \
#   --synthetic-dialogues 16 \
#   --utterances-per-dialogue 8 \
#   --splits train dev test

In [ ]:
# Copy cache to local runtime disk for faster training I/O.
!rm -rf /content/facepred_cache
!mkdir -p /content/facepred_cache
!rsync -a "$DRIVE_ROOT/cache/$CACHE_NAME/" /content/facepred_cache/

In [ ]:
!python scripts/train_world_model.py \
  --config configs/config.yaml \
  --cache-dir /content/facepred_cache \
  --output-dir "$DRIVE_ROOT/runs/$RUN_NAME" \
  --epochs 10 \
  --batch-size 32 \
  --device auto \
  --amp \
  --resume auto \
  --save-every-steps 250

In [ ]:
!python scripts/evaluate_world_model.py \
  --config configs/config.yaml \
  --cache-dir /content/facepred_cache \
  --checkpoint "$DRIVE_ROOT/runs/$RUN_NAME/checkpoints/best.pt" \
  --split test \
  --device auto \
  --output "$DRIVE_ROOT/runs/$RUN_NAME/test_metrics.json"